In [ ]:
from copy import deepcopy

import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging

import pandas as pd
import whisperx
from IPython.display import Audio, display

from drzavni_posao import (
    CharacterAssignerConfig,
    DiarizerConfig,
    EmbedderConfig,
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    SR,
    Stage,
    Transcriber,
    TranscriberConfig,
    run_pipeline,
)
from drzavni_posao.comparison import (
    cast_accuracy,
    compare_all,
    load_transcripts,
    scribe_transcripts,
    text_of,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")

from drzavni_posao.constants import CYRILLIC_TO_LATIN


In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = "/Volumes/luka_erceg/drzavni_posao/audios"
TRANSCRIPT_PATH = (
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)

HF_TOKEN = dbutils.secrets.get(scope="luka-erceg", key="hf_token")

SCRIBE_DIR = f"{TRANSCRIPT_PATH}/elevenlabs"

DEVICE = "cuda"
COMPUTE_TYPE = "bfloat16"
EMBEDDER_MODEL = "pyannote/wespeaker-voxceleb-resnet34-LM"

ASR_MODELS = {
    "sr-cmb": "/Volumes/workspace/default/luka_erceg_models/whisper-sr-ct2",
    "large-v3": "large-v3",
    "large-v3-turbo": "large-v3-turbo",
}
ALIGN_MODELS = {
    "classla": "classla/wav2vec2-xls-r-parlaspeech-hr",
}

CAST_HOTWORDS = (
    "Đorđe Čvarkov, Dragan Torbica, Boškić, Dobrosavljev, "
    "Marijanović, Živan Radosavljević, Frau Šilovička"
)

NGRAM5 = {"no_repeat_ngram_size": 5, "repetition_penalty": 1.0}
DECODING_VARIANTS = {
    "baseline": {},
    "ngram3": {"no_repeat_ngram_size": 3, "repetition_penalty": 1.0},
    "ngram5": NGRAM5,
    "ngram5-numerals": {**NGRAM5, "suppress_numerals": True},
    "ngram5-hotwords": {**NGRAM5, "hotwords": CAST_HOTWORDS},
    "ngram5-beam10": {**NGRAM5, "beam_size": 10},
}

ALIGN_MODEL_KEY = "classla"

RUNS = [
    ("large-v3", "ngram5"),
    ("large-v3", "baseline"),
    ("large-v3", "ngram3"),
    ("large-v3", "ngram5-numerals"),
    ("large-v3", "ngram5-hotwords"),
    ("large-v3", "ngram5-beam10"),
]

EPISODES = set(scribe_transcripts(SCRIBE_DIR))

tracker = MlflowTracker(EXPERIMENT_NAME)


In [ ]:
# Episode 60 alignment hypothesis: source output and affected audio.
DIAGNOSTIC_SOURCE_RUN_ID = "b66150772bc043d091f7b4c9ff606ce0"
DIAGNOSTIC_EPISODE_KEY = "60"
DIAGNOSTIC_CLIPS = [(312.0, 322.0), (322.0, 332.0), (332.0, 342.0)]
DIAGNOSTIC_WINDOW = (DIAGNOSTIC_CLIPS[0][0], DIAGNOSTIC_CLIPS[-1][1])

diagnostic_config = tracker._configuration(DIAGNOSTIC_SOURCE_RUN_ID)
diagnostic_media = MediaStore(
    AUDIO_PATHS,
    "/tmp",
    **diagnostic_config["audio"],
)
diagnostic_episode = diagnostic_media.episode(DIAGNOSTIC_EPISODE_KEY)
diagnostic_source = tracker.episode_run_states(
    DIAGNOSTIC_SOURCE_RUN_ID
)[DIAGNOSTIC_EPISODE_KEY]
tracker.restore_stage(
    diagnostic_episode,
    diagnostic_source.run_id,
    Stage.TRANSCRIPTION,
)
diagnostic_audio = diagnostic_episode.audio()
diagnostic_source_segments = [
    deepcopy(segment)
    for segment in diagnostic_episode.transcript.segments
    if segment["end"] > DIAGNOSTIC_WINDOW[0]
    and segment["start"] < DIAGNOSTIC_WINDOW[1]
]

print("Source-run text:")
print(" ".join(segment["text"].strip() for segment in diagnostic_source_segments))
display(
    Audio(
        diagnostic_audio[
            int(DIAGNOSTIC_WINDOW[0] * SR) : int(DIAGNOSTIC_WINDOW[1] * SR)
        ],
        rate=SR,
    )
)


In [ ]:
# Run the exact source ASR configuration on the full episode, without alignment.
diagnostic_transcriber = Transcriber(
    TranscriberConfig(**diagnostic_config["transcription"])
)
diagnostic_full_raw = diagnostic_transcriber.asr.transcribe(
    diagnostic_audio,
    batch_size=diagnostic_transcriber.config.batch_size,
)
for segment in diagnostic_full_raw["segments"]:
    segment["text"] = segment["text"].translate(CYRILLIC_TO_LATIN)

diagnostic_full_raw_segments = [
    segment
    for segment in diagnostic_full_raw["segments"]
    if segment["end"] > DIAGNOSTIC_WINDOW[0]
    and segment["start"] < DIAGNOSTIC_WINDOW[1]
]
assert diagnostic_full_raw_segments

display(
    pd.DataFrame(
        [
            {
                "start": segment["start"],
                "end": segment["end"],
                "avg_logprob": segment.get("avg_logprob"),
                "text": segment["text"].strip(),
            }
            for segment in diagnostic_full_raw_segments
        ]
    )
)

In [ ]:
# Re-run only the bad 30-second region as three independent 10-second clips.
def shift_timestamps(segments, offset):
    for segment in segments:
        for key in ("start", "end"):
            if key in segment:
                segment[key] += offset
        for word in segment.get("words", []):
            for key in ("start", "end"):
                if key in word:
                    word[key] += offset


def transcribe_clip(transcriber, start, end):
    clip = diagnostic_audio[int(start * SR) : int(end * SR)]
    raw = transcriber.asr.transcribe(
        clip,
        batch_size=transcriber.config.batch_size,
    )
    for segment in raw["segments"]:
        segment["text"] = segment["text"].translate(CYRILLIC_TO_LATIN)

    aligned = whisperx.align(
        deepcopy(raw["segments"]),
        transcriber.align_model,
        transcriber.align_meta,
        clip,
        transcriber.config.device,
        return_char_alignments=False,
    )
    raw_segments = deepcopy(raw["segments"])
    aligned_segments = aligned["segments"]
    shift_timestamps(raw_segments, start)
    shift_timestamps(aligned_segments, start)
    return {"clip": (start, end), "raw": raw_segments, "aligned": aligned_segments}


try:
    diagnostic_retries = [
        transcribe_clip(diagnostic_transcriber, start, end)
        for start, end in DIAGNOSTIC_CLIPS
    ]
finally:
    diagnostic_transcriber.free()

print(f"Finished {len(diagnostic_retries)} short-window retries.")

In [ ]:
# Compare transcript text and the long-word failure signal.
diagnostic_retry_segments = [
    segment
    for retry in diagnostic_retries
    for segment in retry["aligned"]
]

for retry in diagnostic_retries:
    start, end = retry["clip"]
    print(f"\nRaw ASR {start:.0f}–{end:.0f}s:")
    print(" ".join(segment["text"].strip() for segment in retry["raw"]))


def word_rows(mode, segments):
    return [
        {
            "mode": mode,
            "start": word["start"],
            "end": word["end"],
            "duration": word["end"] - word["start"],
            "score": word.get("score"),
            "word": word["word"],
            "segment": segment["text"].strip(),
        }
        for segment in segments
        for word in segment.get("words", [])
        if "start" in word and "end" in word
    ]


diagnostic_words = pd.DataFrame(
    word_rows("source", diagnostic_source_segments)
    + word_rows("short retry", diagnostic_retry_segments)
)
assert len(diagnostic_retries) == len(DIAGNOSTIC_CLIPS)
assert not diagnostic_words.empty

diagnostic_summary = pd.DataFrame(
    [
        {
            "mode": mode,
            "words": len(group),
            "max_word_seconds": group["duration"].max(),
            "words_over_2s": (group["duration"] > 2).sum(),
            "mean_alignment_score": group["score"].mean(),
        }
        for mode, group in diagnostic_words.groupby("mode")
    ]
).set_index("mode")

display(diagnostic_summary)
display(
    diagnostic_words[diagnostic_words["duration"] >= 1.5]
    .sort_values(["mode", "start"])
    .reset_index(drop=True)
)
print(
    "Hypothesis supported if the short-window raw text contains audible words "
    "missing from the source text and its long-word spans shrink."
)

In [ ]:
# Find source-run episodes at least as low-confidence as episode 60.
import json
from pathlib import Path
from tempfile import TemporaryDirectory

parent = tracker.client.get_run(DIAGNOSTIC_SOURCE_RUN_ID)
episode_runs = tracker.client.search_runs(
    [parent.info.experiment_id],
    filter_string=(
        f"tags.mlflow.parentRunId = '{DIAGNOSTIC_SOURCE_RUN_ID}' "
        "AND tags.run_level = 'episode'"
    ),
    max_results=1000,
)

confidence = pd.DataFrame(
    [
        {
            "episode": run.data.tags["episode_key"],
            "run_id": run.info.run_id,
            "p10": run.data.metrics.get("transcription.alignment_score_p10"),
            "median": run.data.metrics.get("transcription.alignment_score_median"),
        }
        for run in episode_runs
        if "episode_key" in run.data.tags
    ]
).dropna()
episode_60 = confidence[confidence["episode"] == DIAGNOSTIC_EPISODE_KEY].iloc[0]
candidates = confidence[
    (confidence["p10"] <= episode_60["p10"])
    | (confidence["median"] <= episode_60["median"])
].copy()

stretch = []
for run_id in candidates["run_id"]:
    with TemporaryDirectory() as directory:
        path = tracker.client.download_artifacts(
            run_id,
            "transcription/transcript.json",
            directory,
        )
        transcript = json.loads(Path(path).read_text(encoding="utf-8"))
    durations = [
        word["end"] - word["start"]
        for segment in transcript["segments"]
        for word in segment.get("words", [])
        if "start" in word and "end" in word
    ]
    stretch.append(
        {
            "max_word_seconds": max(durations, default=0),
            "words_over_2s": sum(duration > 2 for duration in durations),
        }
    )

candidates[["max_word_seconds", "words_over_2s"]] = pd.DataFrame(
    stretch,
    index=candidates.index,
)
print(
    f"{len(candidates)} of {len(confidence)} episodes are at least as low-confidence "
    "as episode 60 on p10 or median alignment score."
)
display(
    candidates.sort_values(["median", "p10"])[
        ["episode", "p10", "median", "max_word_seconds", "words_over_2s", "run_id"]
    ].reset_index(drop=True)
)